# ST-GCN joint_bone - leave-one-out augmentation ablation

"advanced" augmentation has only ever been tested as one fixed bundle of 4 techniques (`aug_bone_rotation`, `aug_confidence_scaled_noise`, `aug_joint_masking`, `aug_temporal_warp`) - never isolated to see which ones actually help versus drag the bundle down. Working hypothesis: `aug_joint_masking` (randomly zeroing out joints, a simulated-occlusion technique) may be actively hurting here - badminton shot classification hinges heavily on precise arm/racket position, and randomly blinding the model to that during training could remove exactly the signal it needs most.

Five configs, same architecture/schedule/weight_decay as the proven winning recipe (`joint_bone`, `weight_decay=1e-3`, StepLR, sampler built fresh per call, seeded workers), back on the 30-frame data (the 50-frame re-extraction showed no real benefit):
- **full_advanced** - all 4 techniques, the existing baseline/control
- **no_bone_rotation**, **no_confidence_scaled_noise**, **no_joint_masking**, **no_temporal_warp** - each with exactly one technique removed

**This is a single-seed screening pass**, not a final trustworthy result - given today's noise-floor findings, whichever variant looks best here should get a proper multi-seed confirmation run before being treated as a real improvement, the same pattern used for the regularizer ablation earlier.

### determinism - before torch touches CUDA

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score,
    confusion_matrix, precision_recall_fscore_support,
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

import mmaction.models  # noqa: F401
from mmaction.utils import register_all_modules
register_all_modules()
from mmaction.registry import MODELS

from clearml import Task

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### DataLoader worker-seeding fix

In [2]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


### paths

In [3]:
OUT_ROOT = "/workspace/outputs/stgcn_aug_ablation"
CKPT_ROOT = "/workspace/checkpoints/stgcn_aug_ablation"
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)


### data - back to the 30-frame extraction

The base `train_ds` here uses `augmentation_policy=None` - each leave-one-out variant gets its own dataset instance built inside the training function instead, since the policy itself is what's being varied per config.

In [4]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

train_manifest = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_manifest = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)

KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"

val_ds = ds.BadmintonPoseDataset(val_manifest, KEYPOINTS_ROOT, class_to_id)
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, train {len(train_manifest)}, val {len(val_ds)}, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, train 6251, val 781, test 782


### register the leave-one-out augmentation policies

`ds.ADVANCED_AUGMENTATIONS` is the real, confirmed list of 4 functions (`aug_bone_rotation`, `aug_confidence_scaled_noise`, `aug_joint_masking`, `aug_temporal_warp`) - verified via grep before this was built, not assumed. Each variant filters by `.__name__`, a standard Python attribute, not guessed dataset.py internals.

In [5]:
technique_names = [fn.__name__ for fn in ds.ADVANCED_AUGMENTATIONS]
print("techniques in ADVANCED_AUGMENTATIONS:", technique_names)

ds.AUGMENTATION_POLICIES["full_advanced_loo"] = list(ds.ADVANCED_AUGMENTATIONS)
for name in technique_names:
    policy_name = f"no_{name.replace('aug_', '')}"
    ds.AUGMENTATION_POLICIES[policy_name] = [
        fn for fn in ds.ADVANCED_AUGMENTATIONS if fn.__name__ != name
    ]
    print(f"registered {policy_name}: {[fn.__name__ for fn in ds.AUGMENTATION_POLICIES[policy_name]]}")


techniques in ADVANCED_AUGMENTATIONS: ['aug_bone_rotation', 'aug_confidence_scaled_noise', 'aug_joint_masking', 'aug_temporal_warp']
registered no_bone_rotation: ['aug_confidence_scaled_noise', 'aug_joint_masking', 'aug_temporal_warp']
registered no_confidence_scaled_noise: ['aug_bone_rotation', 'aug_joint_masking', 'aug_temporal_warp']
registered no_joint_masking: ['aug_bone_rotation', 'aug_confidence_scaled_noise', 'aug_temporal_warp']
registered no_temporal_warp: ['aug_bone_rotation', 'aug_confidence_scaled_noise', 'aug_joint_masking']


### batch -> ST-GCN input - joint_bone, in_channels=5

In [6]:
def batch_to_stgcn_input(batch):
    x = torch.cat([batch["keypoints"].float(), batch["bone"].float()], dim=-1)
    return x.unsqueeze(1).unsqueeze(1)


### SMOKE TEST - run alone first

In [7]:
smoke_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy="full_advanced_loo")
smoke_loader = DataLoader(smoke_ds, batch_size=4, shuffle=True)
smoke_batch = next(iter(smoke_loader))
smoke_input = batch_to_stgcn_input(smoke_batch)
print("input shape:", smoke_input.shape)
assert smoke_input.shape[-1] == 5, f"expected 5 channels, got {smoke_input.shape[-1]}"
print("smoke test passed")


input shape: torch.Size([4, 1, 1, 30, 17, 5])
smoke test passed


### metrics

In [8]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for batch in loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)
        logits = model(x, mode="tensor", stage="head")
        probs = torch.softmax(logits, dim=1)

        all_labels.append(y.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels)
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)

    return {
        "top1": accuracy_score(labels, preds),
        "top5": top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes))),
        "mca": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "labels": labels, "preds": preds,
    }


### the reusable training function

Fixed: joint_bone architecture, weight_decay=1e-3, StepLR, seeded workers, fresh sampler per call. Only `augmentation_policy` varies between calls.

In [9]:
def train_one_policy(policy_name, seed=42, step_size=10, num_epochs=60, patience=20,
                      batch_size=16, lr=0.01, weight_decay=1e-3):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    train_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy=policy_name)
    balanced_sampler = ds.build_class_balanced_sampler(train_manifest, target_per_class=500, seed=seed)

    model_cfg = dict(
        type="RecognizerGCN",
        backbone=dict(
            type="STGCN",
            graph_cfg=dict(layout="coco", mode="stgcn_spatial"),
            in_channels=5, base_channels=64, ch_ratio=2,
            num_stages=9, inflate_stages=[4, 7], down_stages=[4, 7],
        ),
        cls_head=dict(type="GCNHead", num_classes=num_classes, in_channels=256,
                      loss_cls=dict(type="CrossEntropyLoss")),
    )
    model = MODELS.build(model_cfg).to(device)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=balanced_sampler,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=seed_worker, generator=loader_generator,
    )

    task = Task.init(project_name="BadmintonPoseAnalysis", task_name=f"stgcn_aug_ablation_{policy_name}", reuse_last_task_id=False)
    logger = task.get_logger()
    task.connect({
        "policy_name": policy_name, "stream": "joint_bone", "seed": seed,
        "lr": lr, "weight_decay": weight_decay, "num_epochs": num_epochs,
        "batch_size": batch_size,
    })

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=0.1)

    best_val_top1 = 0.0
    best_epoch = -1
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_labels, epoch_preds = [], []

        for batch in train_loader:
            x = batch_to_stgcn_input(batch).to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(x, mode="tensor", stage="head")
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_labels.append(y.cpu().numpy())
            epoch_preds.append(logits.argmax(dim=1).cpu().numpy())

        scheduler.step()

        train_loss = epoch_loss / len(train_loader.dataset)
        train_acc = accuracy_score(np.concatenate(epoch_labels), np.concatenate(epoch_preds))
        val_metrics = evaluate(model, val_loader)

        logger.report_scalar("loss", "train", train_loss, iteration=epoch)
        logger.report_scalar("accuracy", "train", train_acc, iteration=epoch)
        logger.report_scalar("accuracy", "val_top1", val_metrics["top1"], iteration=epoch)
        logger.report_scalar("accuracy", "val_mca", val_metrics["mca"], iteration=epoch)

        if val_metrics["top1"] > best_val_top1:
            best_val_top1 = val_metrics["top1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(f"[{policy_name}] epoch {epoch:3d}  train_acc={train_acc:.4f}  "
                  f"val_top1={val_metrics['top1']:.4f}  val_mca={val_metrics['mca']:.4f}")

        if epochs_without_improvement >= patience:
            print(f"[{policy_name}] early stopping at epoch {epoch}, best val_top1={best_val_top1:.4f} (epoch {best_epoch})")
            break

    model.load_state_dict(best_state)

    train_final = evaluate(model, DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True))
    test_final = evaluate(model, test_loader)

    metrics = {
        "policy_name": policy_name,
        "best_epoch": best_epoch,
        "train_acc": train_final["top1"],
        "test_top1": test_final["top1"],
        "test_top5": test_final["top5"],
        "test_mca": test_final["mca"],
        "test_macro_f1": test_final["macro_f1"],
        "train_test_gap": train_final["top1"] - test_final["top1"],
    }

    for metric_name in ["train_acc", "test_top1", "test_top5", "test_mca", "test_macro_f1", "train_test_gap", "best_epoch"]:
        logger.report_single_value(name=metric_name, value=metrics[metric_name])

    cm = confusion_matrix(test_final["labels"], test_final["preds"], labels=list(range(num_classes)))
    logger.report_confusion_matrix(
        title="Test Confusion Matrix", series=policy_name,
        matrix=cm, xaxis="Predicted", yaxis="Actual",
        xlabels=class_names, ylabels=class_names,
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        test_final["labels"], test_final["preds"], labels=list(range(num_classes)), zero_division=0,
    )
    per_class_df = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    logger.report_table(title="Per-Class Test Metrics", series=policy_name, table_plot=per_class_df)

    ckpt_path = os.path.join(CKPT_ROOT, f"{policy_name}_best.pt")
    torch.save(model.state_dict(), ckpt_path)
    task.upload_artifact("checkpoint", ckpt_path)
    per_class_df.to_csv(os.path.join(OUT_ROOT, f"per_class_{policy_name}.csv"), index=False)
    task.close()

    return metrics


### run all five policies

5x a single run - screening pass, single seed (42) throughout. Whichever comes out ahead gets a proper multi-seed confirmation afterward, not treated as final from this alone.

In [10]:
policies_to_test = ["full_advanced_loo"] + [f"no_{n.replace('aug_', '')}" for n in technique_names]
print("testing:", policies_to_test)

all_results = {}

for policy_name in policies_to_test:
    print(f"\n=== training policy={policy_name} ===")
    metrics = train_one_policy(policy_name)
    all_results[policy_name] = metrics
    print(f"[{policy_name}] FINAL: test_top1={metrics['test_top1']:.4f}  "
          f"gap={metrics['train_test_gap']:.4f}  test_mca={metrics['test_mca']:.4f}  "
          f"test_macro_f1={metrics['test_macro_f1']:.4f}")


testing: ['full_advanced_loo', 'no_bone_rotation', 'no_confidence_scaled_noise', 'no_joint_masking', 'no_temporal_warp']

=== training policy=full_advanced_loo ===
ClearML Task: created new task id=8b407d58bcb34864a154326811342bde
2026-09-10 15:15:36,880 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/8b407d58bcb34864a154326811342bde/output/log
[full_advanced_loo] epoch   0  train_acc=0.2898  val_top1=0.5506  val_mca=0.3714
[full_advanced_loo] epoch  10  train_acc=0.7645  val_top1=0.6761  val_mca=0.6453
[full_advanced_loo] epoch  20  train_acc=0.8910  val_top1=0.7093  val_mca=0.6425
[full_advanced_loo] epoch  30  train_acc=0.9142  val_top1=0.7196  val_mca=0.6293
[full_advanced_loo] epoch  40  train_acc=0.9167  val_top1=0.7196  val_mca=0.6348
[full_advanced_loo] early stopping at epoch 47, best val_top1=0.7324 (epoch 27)


███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  7.24MB/s]: 


[full_advanced_loo] FINAL: test_top1=0.7110  gap=0.2161  test_mca=0.6160  test_macro_f1=0.6302

=== training policy=no_bone_rotation ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=6aceaec987d44fc282c2d64c8f20d7e8
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/6aceaec987d44fc282c2d64c8f20d7e8/output/log
[no_bone_rotation] epoch   0  train_acc=0.2881  val_top1=0.4738  val_mca=0.3709
[no_bone_rotation] epoch  10  train_acc=0.7732  val_top1=0.6889  val_mca=0.6447
[no_bone_rotation] epoch  20  train_acc=0.8923  val_top1=0.7183  val_mca=0.6540
[no_bone_rotation] epoch  30  train_acc=0.9215  val_top1=0.7145  val_mca=0.6321
[no_bone_rotation] epoch  40  train_acc=0.9236  val_top1=0.7157  val_mca=0.6389
[no_bone_rotation] epoch  50  train_acc=0.9255  val_top1=0.7222  val_mca=0.6503
[no_bone_rotation] epoch  59  train_acc=0.9227  val_top1=0.7055  val_mca=0.6324


...

[no_joint_masking] FINAL: test_top1=0.7008  gap=0.2196  test_mca=0.6108  test_macro_f1=0.6256

=== training policy=no_temporal_warp ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=246eab6030024e23998897f9e305167e
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/246eab6030024e23998897f9e305167e/output/log
[no_temporal_warp] epoch   0  train_acc=0.2864  val_top1=0.5378  val_mca=0.3511
[no_temporal_warp] epoch  10  train_acc=0.7812  val_top1=0.6825  val_mca=0.6382
[no_temporal_warp] epoch  20  train_acc=0.9095  val_top1=0.7209  val_mca=0.6551
[no_temporal_warp] epoch  30  train_acc=0.9343  val_top1=0.7273  val_mca=0.6390
[no_temporal_warp] epoch  40  train_acc=0.9382  val_top1=0.7247  val_mca=0.6476
[no_temporal_warp] epoch  50  train_acc=0.9384  val_top1=0.7298  val_mca=0.6501
[no_temporal_warp] early stopping at epoch 54, best val_top1=0.7388 (epoch 34)


███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  6.96MB/s]: 


[no_temporal_warp] FINAL: test_top1=0.7059  gap=0.2466  test_mca=0.6017  test_macro_f1=0.6138


### comparison table

In [11]:
LI_ET_AL_BENCHMARK = {"policy_name": "Li et al. benchmark", "best_epoch": None,
                      "test_top1": 0.7441, "test_top5": 0.9376, "test_mca": 0.6144,
                      "test_macro_f1": None, "train_test_gap": None}

comparison = pd.DataFrame(list(all_results.values()) + [LI_ET_AL_BENCHMARK]).set_index("policy_name")
comparison = comparison[["best_epoch", "train_acc", "test_top1", "train_test_gap",
                          "test_top5", "test_mca", "test_macro_f1"]]
comparison.to_csv(os.path.join(OUT_ROOT, "augmentation_loo_comparison.csv"))
comparison


,best_epoch,train_acc,test_top1,train_test_gap,test_top5,test_mca,test_macro_f1
policy_name,,,,,,,
full_advanced_loo,27.0,0.927052,0.710997,0.216054,0.945013,0.615995,0.630229
no_bone_rotation,43.0,0.940969,0.704604,0.236366,0.939898,0.599593,0.611548
no_confidence_scaled_noise,17.0,0.876500,0.689258,0.187241,0.942455,0.600114,0.601543
no_joint_masking,15.0,0.920333,0.700767,0.219565,0.941176,0.610842,0.625562
no_temporal_warp,34.0,0.952488,0.705882,0.246605,0.945013,0.601714,0.613838
Li et al. benchmark,NaN,NaN,0.744100,NaN,0.937600,0.614400,NaN


---

Compare each `no_X` row against `full_advanced_loo` - a variant that clearly beats the full bundle means that technique was net-negative and worth dropping. If `no_joint_masking` comes out ahead, that confirms the hypothesis above. Given this is single-seed, treat any difference smaller than roughly 1pp as "maybe" rather than "confirmed" - the next step for whichever variant looks genuinely better is a 3-seed confirmation run, same pattern as the multi-seed baseline notebooks.